<a href="https://colab.research.google.com/github/velasquezcalderonrafael32-git/Proyecto-Final-Programacion-Rafael-Velasquez-y-Juan-Martin-Franco/blob/Rama-JM/Proyecto_Final_Programaci%C3%B3n_Rafael_Velasquez_y_Juan_Martin_Franco.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

JUAN MARTIN FRANCO - RAFAEL VELASQUEZ

---



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

datos_panaderia = pd.ExcelFile("datos_panaderia.xlsx")

productos = pd.read_excel(datos_panaderia, sheet_name="Productos").dropna(subset=["Categoría"])
zonas = pd.read_excel(datos_panaderia, sheet_name="Zonas").dropna(subset=["Distancia al negocio (km)"])
parametros = pd.read_excel(datos_panaderia, sheet_name="Parametros")
franjas = pd.read_excel(datos_panaderia, sheet_name="Franjas").dropna(subset=["Horario"])
pedidos = pd.read_excel(datos_panaderia, sheet_name="Pedidos").dropna(subset=["Zona"])
escenarios = pd.read_excel(datos_panaderia, sheet_name="Escenarios")
print(len(productos), "productos |", len(zonas), "zonas |", len(pedidos), "pedidos |", len(escenarios), "escenarios")

10 productos | 8 zonas | 80 pedidos | 6 escenarios


In [ ]:
valores = parametros["Valor"].tolist()
p = {
    "costo_km":   valores[0],
    "pago_hora":  valores[1],
    "empaque":    valores[2],
    "comision":   valores[3],
    "n_rep":      int(valores[4]),
    "prep":       valores[6],
    "espera_max": valores[7],
    "margen":     valores[8],
}
print(p)

tiempo_zona = dict(zip(zonas["Zona (barrio)"], zonas["Tiempo ida y vuelta (min)"]))
dist_zona   = dict(zip(zonas["Zona (barrio)"], zonas["Distancia al negocio (km)"]))

{'costo_km': 700.0, 'pago_hora': 12000.0, 'empaque': 800.0, 'comision': 0.0, 'n_rep': 2, 'prep': 12.0, 'espera_max': 45.0, 'margen': 0.62}


In [ ]:
def costo_reparto(distancia_km, tiempo_min, valor, p):
    """Costo de entregar un pedido (en COP)."""
    km = distancia_km * 2 * p["costo_km"]            # ida y vuelta
    mano_obra = tiempo_min / 60 * p["pago_hora"]
    comision = valor * p["comision"]
    return km + mano_obra + p["empaque"] + comision


def ganancia_pedido(valor, tarifa, costo, p):
    """Margen de los productos + tarifa cobrada - costo de reparto."""
    return valor * p["margen"] + tarifa - costo


# Se calcula el costo de reparto y la ganancia de cada pedido usando las funciones definidas arriba
costos, ganancias = [], []
for _, fila in pedidos.iterrows():
    zona = fila["Zona"]
    c = costo_reparto(dist_zona[zona], tiempo_zona[zona], fila["Valor productos (COP)"], p)
    g = ganancia_pedido(fila["Valor productos (COP)"], fila["Tarifa cobrada (COP)"], c, p)
    costos.append(c)
    ganancias.append(g)

pedidos["Costo reparto"] = costos
pedidos["Ganancia"] = ganancias
pedidos["Rentable"] = pedidos["Ganancia"] > 0

print("Pedidos rentables:", (pedidos['Rentable']).sum(), "de", len(pedidos))
pedidos.head()

Pedidos rentables: 60 de 80


,ID pedido,Día,Franja,Zona,N.º productos,Valor productos (COP),Distancia (km),Tarifa cobrada (COP),Costo reparto,Ganancia,Rentable
0,P001,Sábado,Mañana,Chapinero Central,1.0,5500.0,2.5,3500.0,9100.0,-2190.0,False
1,P002,Martes,Noche,Teusaquillo,1.0,116000.0,4.0,4500.0,13200.0,63220.0,True
2,P003,Sábado,Mediodía,Chapinero Alto,2.0,18800.0,1.5,3000.0,6500.0,8156.0,True
3,P004,Domingo,Mediodía,Teusaquillo,1.0,3000.0,4.0,4500.0,13200.0,-6840.0,False
4,P005,Miércoles,Mañana,Chapinero Alto,5.0,42300.0,1.5,3000.0,6500.0,22726.0,True
